In [1]:
# Standard Library
import os
import json
import pickle
import imaplib
import email
from email.header import decode_header
from typing import TypedDict

# Google OAuth Libraries
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# LangChain
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode

load_dotenv()

True

In [2]:
class ChatState(TypedDict):
    messages: list

In [3]:
IMAP_HOST = os.getenv("IMAP_HOST")
IMAP_USER = os.getenv("IMAP_USER")
IMAP_PASSWORD = os.getenv("IMAP_PASSWORD")
IMAP_FOLDER = "INBOX"

print("HOST: ",IMAP_HOST)
print("USER:",IMAP_USER)

HOST:  imap.gmail.com
USER: amithvardhangd3@gmail.com


##### 1. Connect to GMail IMAP

In [4]:
SCOPES = ["https://mail.google.com/"]

TOKEN_FILE = "token.pickle"
CREDENTIALS_FILE = "src/credentials.json"

# Fetching the OAuth credentials
def get_credentials():  
    creds = None  
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE,"rb") as fp:
            creds = pickle.load(fp)
    if creds:
        if creds.expired:
            if creds.refresh_token:
                try:
                    creds.refresh(Request())
                except Exception:
                    creds = None
            else:
                creds = None
    
    if not creds:
        flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE,SCOPES)
        creds = flow.run_local_server(port=0)
        with open(TOKEN_FILE,"wb") as fp:
            pickle.dump(creds,fp)
    return creds

# Using the OAuth credentials, Gmail's IMAP host and the username to login to the Gmail IMAP
def connect():
    creds = get_credentials()
    mail = imaplib.IMAP4_SSL(IMAP_HOST)
    authorisation_string = f"user={IMAP_USER}\1auth=Bearer {creds.token}\1\1"
    mail.authenticate("XOAUTH2",lambda x: authorisation_string)
    return mail
"""
mail = connect()
status, folders = mail.list()
print(status)
for f in folders:
    print(f.decode())
"""

'\nmail = connect()\nstatus, folders = mail.list()\nprint(status)\nfor f in folders:\n    print(f.decode())\n'

##### 2. Write a function to list all the unread messages from the GMail Inbox, list them; and make it a tool

In [5]:
@tool
def list_unread_emails():
    """Returns all the unread emails with UID, Subject, Date, Sender from the inbox"""
    print("List Unread Emails tool called")
    conn = connect()
    conn.select("INBOX")
    status, data = conn.search(None, "UNSEEN")


    list_of_emails = []

    for i in data[0].split():
        status, msg = conn.fetch(i,"(BODY.PEEK[])")
        if status != "OK":
            continue
        raw_email = msg[0][1]
        email_message = email.message_from_bytes(raw_email)

        # Decode the subject of the email
        subject, encoding = decode_header(email_message.get("Subject"))[0]
        if isinstance(subject, bytes):
            subject = subject.decode(encoding or "utf-8", errors="ignore")

        sender = email_message.get("From")
        date = email_message.get("Date")

        list_of_emails.append({
            "uid": i.decode(), #Message serial number according to IMAP
            "date": date,#.astimezone().strftime("%Y-%m-%d %H:%M"),
            "subject": subject,
            "sender": sender
        })

    return json.dumps(list_of_emails, indent=2)

##### 3. Function that summarises the content of an email, given its UID

In [6]:
def prompt_for_summarise_email(uid):
    """Frames a prompt required to summarise the content from an email given its UID"""
    conn = connect()
    conn.select("INBOX")
    #status, data = conn.search(None, "UNSEEN")
    
    status, msg = conn.fetch(uid,"(BODY.PEEK[])")
    if status != "OK":
        return None
    raw_email = msg[0][1]
    email_message = email.message_from_bytes(raw_email)

    # Decode the subject of the email
    subject, encoding = decode_header(email_message.get("Subject"))[0]
    if isinstance(subject, bytes):
        subject = subject.decode(encoding or "utf-8", errors="ignore")

    sender = email_message.get("From")
    date = email_message.get("Date")

    body = ""
    if email_message.is_multipart():
        for part in email_message.walk():
            content_type = part.get_content_type()
            disposition = str(part.get("Content-Disposition"))
            if(content_type=="text/plain" and "attachment" not in disposition):
                payload = part.get_payload(decode=True)
                if payload:
                    body += payload.decode(errors="ignore") + "\n"
                    #break
    else:
        payload = email_message.get_payload(decode=True)
        if payload:
            body = payload.decode(errors="ignore")

    prompt = (
        f"Summarise this email concisely:\n"
        f"Subject: {subject}\n"
        f"Date: {date}\n"
        f"Sender: {sender}\n"
        f"{body}"
    )
    
    return prompt

@tool
def summarise_email(uid):
    """Returns the summary of an email given its UID"""
    print("Summarise Email tool called")
    prompt = prompt_for_summarise_email(uid)
    return raw_llm.invoke(prompt).content

#### 4. Instantiate the LLM and bind tools with it

In [7]:
raw_llm = init_chat_model(model="qwen3.5:0.8b",model_provider="ollama")
llm = raw_llm.bind_tools([list_unread_emails,summarise_email])

In [93]:
raw_llm.invoke("What is Ollama?").content

KeyboardInterrupt: 

#### Define the LLM node

In [8]:
def llm_node(state):
    response = llm.invoke(state["messages"])
    return {
        "messages" : state["messages"] + [response]
    }

#### Define the router node

In [9]:
def router(state):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    else:
        return "end"

#### Define the tool node

In [10]:
"""
def tool_node(state):
    result = tool_node.invoke(state)
    dic = {
        "messages" : state["messages"] + result["messages"]
    }

    return dic
"""
tool_node = ToolNode([list_unread_emails,summarise_email])

#### Now, construct the agentic workflow

In [11]:
builder = StateGraph(ChatState)

builder.add_node("llm",llm_node)
builder.add_node("tools",tool_node)

builder.add_edge(START,"llm")
builder.add_edge("tools","llm")

builder.add_conditional_edges("llm",router,{"tools":"tools","end":END})

graph = builder.compile()

#### Write the code that uses the above agentic workflow

In [117]:
state = {
    "messages" : []
}

user_prompt = "Can you please retrieve latest 10 unread emails?"

state["messages"].append({"role":"user","content":user_prompt})
response = graph.invoke(state)
state["messages"].append({"role":"assistant","content":response["messages"][-1].content})
print(response["messages"][-1].content)

KeyboardInterrupt: 

In [15]:
SYSTEM_PROMPT = """
    You are a professional AI Email Assistant whose primary responsibility is helping users efficiently manage their email inbox.

    Your objectives are to:

    List unread emails.
    Summarize individual emails.
    Provide accurate, concise, and professional responses.

    GENERAL BEHAVIOR

    Always be helpful, concise, and professional.
    Think through the user's request before deciding which tool(s) to use.
    Never guess email information. Use the available tools whenever email data is required.
    Base all responses only on information returned by the tools.
    If required information cannot be obtained from the available tools, clearly state the limitation.

    TOOL USAGE

    Available tools:

    list_unread_emails()
    Returns unread emails with:
    UID
    Subject
    Sender
    Date
    summarize_email(uid)
    Returns a concise summary of the specified email.

    TOOL SELECTION RULES

    If the user wants to:
    view unread emails,
    check their inbox,
    see new emails,
    list unread messages,
    or asks a similar request,
    ALWAYS call list_unread_emails() first.
    If the user requests a summary of a specific email:
    Use summarize_email(uid) with the appropriate UID.
    If the UID is not provided but can be identified from a previous tool result, use it.
    Otherwise, ask the user which email they want summarized.
    You may call multiple tools in a single response whenever it improves efficiency.

    RESPONSE FORMAT

    After all necessary tool calls:

    Present the results in a clear, structured format.
    Keep summaries concise while preserving important information.
    Always include the email UID whenever referencing an email.
    Use bullet points or numbered lists where appropriate.

    QUALITY GUIDELINES

    Be accurate and factual.
    Avoid unnecessary verbosity.
    Do not fabricate email details.
    Clearly distinguish between email metadata (sender, subject, date, UID) and summarized content.
    Maintain a professional and user-friendly tone throughout.

    FORMATTING RULES

    Do not use Markdown headings that begin with #, ##, or ###.
    Section titles should be plain bold text only.
    Example: UNREAD EMAILS
    Do not use: # UNREAD EMAILS
    """

In [16]:
state = {
    "messages" : [{"role" : "system", "content" : SYSTEM_PROMPT}]
}

markdown_content = ""
ctr = 1

while True:
    user_prompt = input("> ")
    p = user_prompt.strip().lower()
    if p == "end" or p == "exit" or p == "quit":
        break
    state["messages"].append({"role" : "user", "content" : user_prompt})
    mc_1 = f"### {ctr}. User : "
    markdown_content += (mc_1 + "\n" + user_prompt + "\n\n")
    response = graph.invoke(state)
    res = response["messages"][-1].content
    state["messages"].append({"role" : "assistant", "content" : res})
    mc_2 = f"### {ctr}. Assistant : "
    markdown_content += (mc_2 + "\n" + res + "\n\n\n")
    print(res)
    print()
    ctr += 1

with open("markdown_1.md",mode="w+",encoding="utf-8") as fp:
    fp.write(markdown_content)

An AI agent is an autonomous software entity that simulates intelligent, human-like behavior through specialized knowledge and computational tools. Here's a more detailed explanation:

## Core Characteristics of AI Agents:
1. **Autonomy** - Can make decisions without constant human input
2. **Learning** - Uses data and algorithms to improve over time
3. **Interactivity** - Engages with users in natural, human-like ways
4. **Purpose** - Solves problems, performs tasks, or achieves goals

## Types of AI Agents:
- **Self-driving Agents** - Autonomous vehicles, robotics
- **Natural Language Agents** - Chatbots, speech assistants
- **Predictive Agents** - Systems that forecast future events
- **Specialized Agents** - Medical, legal, and other specialized agents

## Example Use Cases:
- **E-commerce Agents**: Suggest products based on browsing history
- **Customer Support**: Help users with their issues efficiently
- **Research Agents**: Extract information from scientific databases

Would y

In [13]:
"""
state = {
    "messages" : [SystemMessage(content=SYSTEM_PROMPT)]
}

markdown_content = ""
ctr = 1

while True:
    user_prompt = input("> ")
    p = user_prompt.strip().lower()
    if p == "end" or p == "exit" or p == "quit":
        break
    state["messages"].append(HumanMessage(content=user_prompt))
    mc_1 = f"### {ctr}. User : "
    markdown_content += (mc_1 + "\n" + user_prompt + "\n\n")
    response = graph.invoke(state)
    res = response["messages"][-1].content
    state["messages"].append(AIMessage(content=res))
    mc_2 = f"### {ctr}. Assistant : "
    markdown_content += (mc_2 + "\n" + res + "\n\n\n")
    print(res)
    print()
    ctr += 1

with open("markdown_1.md",mode="w+",encoding="utf-8") as fp:
    fp.write(markdown_content)
"""

# Agentic AI

I'd love to help you understand **Agentic AI**! 

## What is Agentic AI?
Agentic AI refers to AI systems that perform actions autonomously rather than just predicting outcomes. They combine:
- **LLMs** for natural language understanding
- **Computer vision** for image processing
- **Machine learning** for pattern recognition
- **Bot execution** for task execution

## How It Works
1. Understand user intent
2. Execute specific actions
3. Return results

## Examples

### Common Use Cases:
- 🛠️ **Autonomous Tasks:** Repairing machinery, organizing files
- 📊 **Data Analysis:** Processing complex datasets without prompts
- 🎯 **Productivity:** Automating repetitive workflows
- 🌍 **Cross-Device:** Handling work across devices and locations

---

## What Would You Like Help With?

1. **Explain Agentic AI concepts**
2. **Get examples of usage**
3. **Create prompts** for using Agentic AI
4. **Ask about automation tools**
5. **Need clarification on a specific use case**

Let me know 

In [14]:
state

{'messages': [SystemMessage(content="\n    You are a professional AI Email Assistant whose primary responsibility is helping users efficiently manage their email inbox.\n\n    Your objectives are to:\n\n    List unread emails.\n    Summarize individual emails.\n    Provide accurate, concise, and professional responses.\n\n    GENERAL BEHAVIOR\n\n    Always be helpful, concise, and professional.\n    Think through the user's request before deciding which tool(s) to use.\n    Never guess email information. Use the available tools whenever email data is required.\n    Base all responses only on information returned by the tools.\n    If required information cannot be obtained from the available tools, clearly state the limitation.\n\n    TOOL USAGE\n\n    Available tools:\n\n    list_unread_emails()\n    Returns unread emails with:\n    UID\n    Subject\n    Sender\n    Date\n    summarize_email(uid)\n    Returns a concise summary of the specified email.\n\n    TOOL SELECTION RULES\n\n   